In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = "/content/drive/MyDrive/DLproject"
print(os.listdir(ROOT))
!pip -q install pysodmetrics
import py_sod_metrics; print("pysodmetrics OK")

Mounted at /content/drive
['edge_sam_best.pth', 'edge_sam_refined_best.pth', 'DL-final.ipynb', 'cached_feats_test.tar.gz', 'cached_feats_train.tar.gz', 'edge_pyramid_sam_final.pth', 'edge_pyramid_sam_best.pth', 'training_and_evaluation_results.txt', 'Untitled7.ipynb']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 15.9 MB/s eta 0:00:00
pysodmetrics OK


In [ ]:
!mkdir -p /content/feats_test /content/feats_train
!tar -xzf {ROOT}/cached_feats_test.tar.gz -C /content/feats_test
!tar -xzf {ROOT}/cached_feats_train.tar.gz -C /content/feats_train

import glob, torch
test_files  = sorted(glob.glob("/content/feats_test/**/*.pt",  recursive=True))
train_files = sorted(glob.glob("/content/feats_train/**/*.pt", recursive=True))
print(len(test_files), len(train_files))
x = torch.load(test_files[0], map_location="cpu")
print(type(x), getattr(x, "shape", None), getattr(x, "dtype", None))
print(test_files[0]); print(test_files[1])


gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now
3195 6000
<class 'torch.Tensor'> torch.Size([256, 64, 64]) torch.float16
/content/feats_test/cached_feats_test/COD10K-CAM-1-Aquatic-1-BatFish-2.pt
/content/feats_test/cached_feats_test/COD10K-CAM-1-Aquatic-1-BatFish-5.pt


In [ ]:
# Cell 2B: Download using the shared Google Drive ID
!pip install -q --upgrade gdown
!mkdir -p /content/dataset
%cd /content/dataset
!gdown 1vRYAie0JcNStcSwagmCq55eirGyMYGm5
!unzip -q *.zip
!rm -f *.zip
%cd /content
print("Dataset downloaded and extracted via gdown!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
/content/dataset
Downloading...
From (original): https://drive.google.com/uc?id=1vRYAie0JcNStcSwagmCq55eirGyMYGm5
From (redirected): https://drive.google.com/uc?id=1vRYAie0JcNStcSwagmCq55eirGyMYGm5&confirm=t&uuid=bb04664e-f3dd-4dc7-a45c-4ce5ae78bd36
To: /content/dataset/COD10K-v3.zip
100% 2.42G/2.42G [01:39<00:00, 24.3MB/s]
/content
Dataset downloaded and extracted via gdown!


In [ ]:
import os, glob

# 1. See what extracted
!find /content/dataset -maxdepth 4 -type d | head -40
def find_dirs(name):
    return [r for r, d, f in os.walk("/content/dataset") if os.path.basename(r) == name]

for name in ["Image", "GT_Object"]:
    for d in find_dirs(name):
        n = len([x for x in os.listdir(d) if x.lower().endswith((".jpg", ".png"))])
        print(f"{name:10s} {n:5d}  {d}")

/content/dataset
/content/dataset/COD10K-v3
/content/dataset/COD10K-v3/Train
/content/dataset/COD10K-v3/Train/Image
/content/dataset/COD10K-v3/Train/GT_Edge
/content/dataset/COD10K-v3/Train/GT_Instance
/content/dataset/COD10K-v3/Train/GT_Object
/content/dataset/COD10K-v3/Info
/content/dataset/COD10K-v3/Test
/content/dataset/COD10K-v3/Test/Image
/content/dataset/COD10K-v3/Test/GT_Edge
/content/dataset/COD10K-v3/Test/GT_Instance
/content/dataset/COD10K-v3/Test/GT_Object
Image       6000  /content/dataset/COD10K-v3/Train/Image
Image       4000  /content/dataset/COD10K-v3/Test/Image
GT_Object   6000  /content/dataset/COD10K-v3/Train/GT_Object
GT_Object   4000  /content/dataset/COD10K-v3/Test/GT_Object


In [ ]:
import os
import glob
import shutil

# 1. Standardize test cache directory
os.makedirs("/content/cached_feats_test", exist_ok=True)

# If files unpacked into the nested folder, move them up
nested_test = "/content/feats_test/cached_feats_test"
if os.path.exists(nested_test):
    print("Moving files from nested folder to /content/cached_feats_test/...")
    !mv /content/feats_test/cached_feats_test/*.pt /content/cached_feats_test/ 2>/dev/null || true

# Count test features available
test_pts = glob.glob("/content/cached_feats_test/*.pt")
print(f"Total test features currently present: {len(test_pts)} / 4000")

Moving files from nested folder to /content/cached_feats_test/...
Total test features currently present: 3195 / 4000


In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import glob
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.transforms as T
from segment_anything import sam_model_registry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_cache_dir = "/content/cached_feats_test"

# Locate Test images
test_img_dir = "/content/dataset/COD10K-v3/Test/Image"
if not os.path.exists(test_img_dir):
    for root, dirs, files in os.walk("/content/dataset"):
        if root.endswith("Test/Image") or root.endswith(os.path.join("Test", "Image")):
            test_img_dir = root
            break

test_imgs = sorted(glob.glob(os.path.join(test_img_dir, "*.[jJ][pP][gG]")))
existing_pts = set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(test_cache_dir, "*.pt")))
missing_imgs = [p for p in test_imgs if os.path.splitext(os.path.basename(p))[0] not in existing_pts]

print(f"Test images missing features: {len(missing_imgs)}")

if len(missing_imgs) > 0:
    sam_checkpoint = "/content/sam_vit_b_01ec64.pth"
    if not os.path.exists(sam_checkpoint):
        import urllib.request
        urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", sam_checkpoint)

    sam_test = sam_model_registry["vit_b"](checkpoint=sam_checkpoint).to(device)
    sam_test.eval()

    transform = T.Compose([
        T.Resize((1024, 1024)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    with torch.no_grad():
        for img_path in tqdm(missing_imgs, desc="Extracting Remaining Test Embeddings"):
            name = os.path.splitext(os.path.basename(img_path))[0]
            out_file = os.path.join(test_cache_dir, f"{name}.pt")
            img = Image.open(img_path).convert("RGB")
            tensor = transform(img).unsqueeze(0).to(device)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                emb = sam_test.image_encoder(tensor)
            torch.save(emb.squeeze(0).half().cpu(), out_file)

    del sam_test
    torch.cuda.empty_cache()

print(f"[✓] Final verified test cache count: {len(glob.glob('/content/cached_feats_test/*.pt'))}")

  Preparing metadata (setup.py) ... done
Test images missing features: 805


Extracting Remaining Test Embeddings: 100%|██████████| 805/805 [03:24<00:00,  3.93it/s]

[✓] Final verified test cache count: 4000


In [ ]:
import os, glob, numpy as np
from PIL import Image

TEST_IMG = "/content/dataset/COD10K-v3/Test/Image"
TEST_GT  = "/content/dataset/COD10K-v3/Test/GT_Object"

if 'test_files' not in globals():
    test_files = sorted(glob.glob("/content/feats_test/**/*.pt", recursive=True))

imgs = sorted(glob.glob(TEST_IMG + "/*.jpg"))
gts  = sorted(glob.glob(TEST_GT + "/*.png"))
print("images:", len(imgs), "| GTs:", len(gts), "| cached feats:", len(test_files))

stem = lambda p: os.path.splitext(os.path.basename(p))[0]
f, i, g = {stem(x) for x in test_files}, {stem(x) for x in imgs}, {stem(x) for x in gts}
print("img∩feat:", len(i & f), "| img missing feat:", len(i - f))
print("img∩gt  :", len(i & g), "| img missing gt  :", len(i - g))

pos = [x for x in gts if np.array(Image.open(x)).max() > 0]
print("positives:", len(pos), "| negatives:", len(gts) - len(pos))

images: 4000 | GTs: 4000 | cached feats: 3195
img∩feat: 3195 | img missing feat: 805
img∩gt  : 4000 | img missing gt  : 0
positives: 2026 | negatives: 1974


In [ ]:
import glob, os, torch
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as T
from segment_anything import sam_model_registry

cache = "/content/cached_feats_test"
stem = lambda p: os.path.splitext(os.path.basename(p))[0]

# 1. scan everything, collect bad files instead of crashing
bad = []
for f in tqdm(sorted(glob.glob(cache + "/*.pt")), desc="Scanning"):
    try:
        x = torch.load(f, map_location="cpu")
        if tuple(x.shape) != (256, 64, 64) or x.dtype != torch.float16 or not torch.isfinite(x).all():
            bad.append(f)
    except Exception:
        bad.append(f)
print("bad files:", len(bad))
for f in bad[:10]: print("  ", os.path.basename(f))

# 2. delete them and re-extract with the same pipeline as before
if bad:
    for f in bad: os.remove(f)
    names = {stem(f) for f in bad}
    redo = [p for p in test_imgs if stem(p) in names]

    sam_fix = sam_model_registry["vit_b"](checkpoint="/content/sam_vit_b_01ec64.pth").to(device).eval()
    tf = T.Compose([T.Resize((1024, 1024)), T.ToTensor(),
                    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    with torch.no_grad():
        for p in tqdm(redo, desc="Re-extracting"):
            t = tf(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                e = sam_fix.image_encoder(t)
            torch.save(e.squeeze(0).half().cpu(), f"{cache}/{stem(p)}.pt")
    del sam_fix; torch.cuda.empty_cache()

# 3. final verification
files = sorted(glob.glob(cache + "/*.pt"))
still_bad = 0
for f in files:
    try:
        x = torch.load(f, map_location="cpu")
        if tuple(x.shape) != (256, 64, 64) or x.dtype != torch.float16 or not torch.isfinite(x).all():
            still_bad += 1
    except Exception:
        still_bad += 1
print("files:", len(files), "| still bad:", still_bad)

Scanning: 100%|██████████| 4000/4000 [01:00<00:00, 65.66it/s]


bad files: 1
   COD10K-NonCAM-4-Terrestial-2780.pt


Re-extracting: 100%|██████████| 1/1 [00:00<00:00,  3.18it/s]


files: 4000 | still bad: 0


In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')

!tar -cf /content/cached_feats_test_full.tar -C /content cached_feats_test
!ls -lh /content/cached_feats_test_full.tar
!cp /content/cached_feats_test_full.tar /content/drive/MyDrive/DLproject/
!tar -tf /content/drive/MyDrive/DLproject/cached_feats_test_full.tar | grep -c "\.pt$"

-rw-r--r-- 1 root root 7.9G Oct 14 07:38 /content/cached_feats_test_full.tar
4000


In [ ]:
import torch, collections
CK = "/content/drive/MyDrive/DLproject/edge_sam_best.pth"
ck = torch.load(CK, map_location="cpu")
print(type(ck))
if isinstance(ck, dict):
    print("top-level keys:", list(ck.keys())[:10])
    for k in ("model_state_dict", "state_dict", "model"):
        if k in ck and isinstance(ck[k], dict):
            ck = ck[k]; print("using nested key:", k); break
sd = ck
groups = collections.OrderedDict()
for k, v in sd.items():
    p = k.split(".")[0]
    n, c = groups.get(p, (0, 0))
    groups[p] = (n + 1, c + v.numel())
for p, (n, c) in groups.items():
    print(f"{p:25s} tensors={n:4d} params={c/1e6:.3f}M")
print("\nnon-decoder keys:")
for k, v in sd.items():
    if "mask_decoder" not in k: print(f"  {k:55s} {tuple(v.shape)}")

<class 'dict'>
top-level keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'loss']
using nested key: model_state_dict
edge_detector             tensors=   2 params=0.000M
adapter                   tensors=   6 params=0.034M
prompt_encoder            tensors=  17 params=0.006M
mask_decoder              tensors= 120 params=4.058M

non-decoder keys:
  edge_detector.sobel_x                                   (1, 1, 3, 3)
  edge_detector.sobel_y                                   (1, 1, 3, 3)
  adapter.down.weight                                     (64, 256, 1, 1)
  adapter.down.bias                                       (64,)
  adapter.edge_proj.weight                                (64, 1, 3, 3)
  adapter.edge_proj.bias                                  (64,)
  adapter.up.weight                                       (256, 64, 1, 1)
  adapter.up.bias                                         (256,)
  prompt_encoder.pe_layer.positional_encoding_gaussian_matrix (2, 128)
  prompt_encode

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from segment_anything import sam_model_registry

device = torch.device("cuda")
SAM_CK = "/content/sam_vit_b_01ec64.pth"

class Sobel(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32)
        self.register_buffer("kx", kx.view(1,1,3,3))
        self.register_buffer("ky", kx.t().contiguous().view(1,1,3,3))
    def forward(self, gray):                      # gray: [B,1,H,W], 0-1
        gx = F.conv2d(gray, self.kx, padding=1)
        gy = F.conv2d(gray, self.ky, padding=1)
        return torch.sqrt(gx**2 + gy**2 + 1e-6)

class EdgeAdapter(nn.Module):
    def __init__(self, c=256, d=64):
        super().__init__()
        self.down = nn.Conv2d(c, d, 1)
        self.edge = nn.Conv2d(1, d, 3, padding=1)
        self.up   = nn.Conv2d(d, c, 1)
        self.act  = nn.GELU()
    def forward(self, feat, edge):
        e = F.interpolate(edge, size=feat.shape[-2:], mode="bilinear", align_corners=False)
        return feat + self.up(self.act(self.down(feat) + self.edge(e)))

class EdgeSAM(nn.Module):
    def __init__(self):
        super().__init__()
        sam = sam_model_registry["vit_b"](checkpoint=SAM_CK)
        self.prompt_encoder, self.mask_decoder = sam.prompt_encoder, sam.mask_decoder
        self.sobel, self.adapter = Sobel(), EdgeAdapter()
    def forward(self, feat, gray):
        x = self.adapter(feat, self.sobel(gray))
        B = x.shape[0]
        sp, de = self.prompt_encoder(points=None, boxes=None, masks=None)
        low, _ = self.mask_decoder(image_embeddings=x,
                                   image_pe=self.prompt_encoder.get_dense_pe(),
                                   sparse_prompt_embeddings=sp.expand(B,-1,-1),
                                   dense_prompt_embeddings=de.expand(B,-1,-1,-1),
                                   multimask_output=False)
        return F.interpolate(low, size=(1024,1024), mode="bilinear", align_corners=False)  # logits

model = EdgeSAM().to(device).eval()
res = model.load_state_dict(sd, strict=False)
print("missing   :", len(res.missing_keys), res.missing_keys[:8])
print("unexpected:", len(res.unexpected_keys), res.unexpected_keys[:8])

missing   : 4 ['sobel.kx', 'sobel.ky', 'adapter.edge.weight', 'adapter.edge.bias']
unexpected: 4 ['edge_detector.sobel_x', 'edge_detector.sobel_y', 'adapter.edge_proj.weight', 'adapter.edge_proj.bias']


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from segment_anything import sam_model_registry

class EdgeDetector(nn.Module):                      # names match checkpoint
    def __init__(self):
        super().__init__()
        self.register_buffer("sobel_x", torch.zeros(1,1,3,3))
        self.register_buffer("sobel_y", torch.zeros(1,1,3,3))
    def forward(self, gray):                        # gray: [B,1,H,W], 0-1
        gx = F.conv2d(gray, self.sobel_x, padding=1)
        gy = F.conv2d(gray, self.sobel_y, padding=1)
        return torch.sqrt(gx**2 + gy**2 + 1e-6)

class EdgeAdapter(nn.Module):
    def __init__(self, c=256, d=64):
        super().__init__()
        self.down = nn.Conv2d(c, d, 1)
        self.edge_proj = nn.Conv2d(1, d, 3, padding=1)
        self.up = nn.Conv2d(d, c, 1)
        self.act = nn.GELU()
    def forward(self, feat, edge):
        e = F.interpolate(edge, size=feat.shape[-2:], mode="bilinear", align_corners=False)
        return feat + self.up(self.act(self.down(feat) + self.edge_proj(e)))

class EdgeSAM(nn.Module):
    def __init__(self):
        super().__init__()
        sam = sam_model_registry["vit_b"](checkpoint=SAM_CK)
        self.prompt_encoder, self.mask_decoder = sam.prompt_encoder, sam.mask_decoder
        self.edge_detector, self.adapter = EdgeDetector(), EdgeAdapter()
    def forward(self, feat, gray):
        x = self.adapter(feat, self.edge_detector(gray))
        B = x.shape[0]
        sp, de = self.prompt_encoder(points=None, boxes=None, masks=None)
        low, _ = self.mask_decoder(image_embeddings=x,
                                   image_pe=self.prompt_encoder.get_dense_pe(),
                                   sparse_prompt_embeddings=sp.expand(B,-1,-1),
                                   dense_prompt_embeddings=de.expand(B,-1,-1,-1),
                                   multimask_output=False)
        return F.interpolate(low, size=(1024,1024), mode="bilinear", align_corners=False)

model = EdgeSAM().to(device).eval()
model.load_state_dict(sd, strict=True)              # raises if anything mismatches
print("loaded OK (strict)")
print("sobel_x:\n", model.edge_detector.sobel_x.flatten().tolist())
print("sobel_y:\n", model.edge_detector.sobel_y.flatten().tolist())
print("edge_proj weight abs-mean:", model.adapter.edge_proj.weight.abs().mean().item())
print("up weight abs-mean       :", model.adapter.up.weight.abs().mean().item())

loaded OK (strict)
sobel_x:
 [-1.0, 0.0, 1.0, -2.0, 0.0, 2.0, -1.0, 0.0, 1.0]
sobel_y:
 [-1.0, -2.0, -1.0, 0.0, 0.0, 0.0, 1.0, 2.0, 1.0]
edge_proj weight abs-mean: 0.17050018906593323
up weight abs-mean       : 0.017911994829773903


In [ ]:
import numpy as np, os, glob, torch
from PIL import Image

stem = lambda p: os.path.splitext(os.path.basename(p))[0]
gt_map = {stem(g): g for g in sorted(glob.glob("/content/dataset/COD10K-v3/Test/GT_Object/*.png"))}
img_map = {stem(p): p for p in test_imgs}
gt_arr = {k: (np.array(Image.open(v)) > 0) for k, v in gt_map.items() if k in list(gt_map)[:0] or True} if False else None

names = sorted(img_map)
pos_n = [n for n in names if np.array(Image.open(gt_map[n])).max() > 0][::100][:10]
neg_n = [n for n in names if np.array(Image.open(gt_map[n])).max() == 0][::200][:5]

def run(n):
    feat = torch.load(f"/content/cached_feats_test/{n}.pt").float().unsqueeze(0).to(device)
    img = Image.open(img_map[n]).convert("RGB").resize((1024, 1024), Image.BILINEAR)
    gray = torch.from_numpy(np.array(img.convert("L"))).float().div(255)[None, None].to(device)
    with torch.no_grad():
        return torch.sigmoid(model(feat, gray))[0, 0].cpu().numpy()

for tag, lst in [("POS", pos_n), ("NEG", neg_n)]:
    for n in lst:
        p = run(n)
        gt = np.array(Image.open(gt_map[n]).resize((1024, 1024), Image.NEAREST)) > 0
        b = p >= 0.5
        iou = (b & gt).sum() / max((b | gt).sum(), 1)
        print(f"{tag} {n[:38]:38s} shape={p.shape} min={p.min():.3f} max={p.max():.3f} "
              f"pred_area={b.mean():.3f} gt_area={gt.mean():.3f} IoU={iou:.3f}")

POS COD10K-CAM-1-Aquatic-1-BatFish-2       shape=(1024, 1024) min=0.001 max=0.999 pred_area=0.168 gt_area=0.216 IoU=0.769
POS COD10K-CAM-1-Aquatic-13-Pipefish-622   shape=(1024, 1024) min=0.000 max=1.000 pred_area=0.113 gt_area=0.081 IoU=0.504
POS COD10K-CAM-1-Aquatic-14-ScorpionFish-8 shape=(1024, 1024) min=0.000 max=0.999 pred_area=0.025 gt_area=0.344 IoU=0.071
POS COD10K-CAM-1-Aquatic-18-StarFish-1181  shape=(1024, 1024) min=0.000 max=0.981 pred_area=0.005 gt_area=0.074 IoU=0.058
POS COD10K-CAM-1-Aquatic-6-Fish-242        shape=(1024, 1024) min=0.000 max=0.999 pred_area=0.031 gt_area=0.047 IoU=0.662
POS COD10K-CAM-2-Terrestrial-22-Bug-1284   shape=(1024, 1024) min=0.000 max=1.000 pred_area=0.027 gt_area=0.027 IoU=0.895
POS COD10K-CAM-2-Terrestrial-23-Cat-1515   shape=(1024, 1024) min=0.000 max=1.000 pred_area=0.075 gt_area=0.081 IoU=0.882
POS COD10K-CAM-2-Terrestrial-28-Deer-1768  shape=(1024, 1024) min=0.000 max=1.000 pred_area=0.180 gt_area=0.184 IoU=0.888
POS COD10K-CAM-2-Terrest

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

OUT = "/content/probs_base"
os.makedirs(OUT, exist_ok=True)

img_dir = "/content/dataset/COD10K-v3/Test/Image"
img_map = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(os.path.join(img_dir, "*.[jJ][pP][gG]"))
}
names = sorted(img_map.keys())
BS = 8

def load_gray(n):
    # Pipeline: RGB -> resize 1024x1024 (bilinear) -> convert("L") -> divide by 255.0
    img = Image.open(img_map[n]).convert("RGB").resize((1024, 1024), Image.BILINEAR)
    return torch.from_numpy(np.array(img.convert("L"), dtype=np.float32)).div(255.0)

def forward_batch_safe(model, feats, grays):
    """
    Batches Sobel and Adapter, runs mask decoder per sample to prevent prompt token mismatch.
    """
    edges = model.edge_detector(grays)
    refined_feats = model.adapter(feats, edges)  # [B, 256, 64, 64]
    sparse_pe, dense_pe = model.prompt_encoder(points=None, boxes=None, masks=None)

    preds = []
    for i in range(refined_feats.shape[0]):
        low_res, _ = model.mask_decoder(
            image_embeddings=refined_feats[i : i + 1],
            image_pe=model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_pe,
            dense_prompt_embeddings=dense_pe,
            multimask_output=False,
        )
        high_res = F.interpolate(low_res, size=(1024, 1024), mode="bilinear", align_corners=False)
        preds.append(high_res)

    return torch.cat(preds, dim=0)

model.eval().to(device)

print(f"Generating probability maps for {len(names)} images...")
with torch.no_grad():
    for i in tqdm(range(0, len(names), BS)):
        b = names[i : i + BS]

        # Resumable skip check
        if all(os.path.exists(f"{OUT}/{n}.npy") for n in b):
            continue

        feat = torch.stack([
            torch.load(f"/content/cached_feats_test/{n}.pt", weights_only=False)
            for n in b
        ]).float().to(device)

        gray = torch.stack([load_gray(n) for n in b]).unsqueeze(1).to(device)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred_logits = forward_batch_safe(model, feat, gray)

        probs = torch.sigmoid(pred_logits)[:, 0].cpu().numpy()

        for n, m in zip(b, probs):
            np.save(f"{OUT}/{n}.npy", (m * 255.0).round().clip(0, 255).astype(np.uint8))

fs = sorted(os.listdir(OUT))
x = np.load(f"{OUT}/{fs[0]}")
print(f"\n[✓] maps: {len(fs)} | sample: {fs[0]} shape: {x.shape} dtype: {x.dtype} min: {x.min()} max: {x.max()}")
!du -sh {OUT}

Generating probability maps for 4000 images...


100%|██████████| 500/500 [02:57<00:00,  2.81it/s]


[✓] maps: 4000 | sample: COD10K-CAM-1-Aquatic-1-BatFish-2.npy shape: (1024, 1024) dtype: uint8 min: 0 max: 255
4.0G	/content/probs_base
